# Fase 6 — Modin · CIC-IoT-2023

600.000 filas, 40 columnas · 10 consultas idénticas por motor. Los resultados se guardan en `06_fase6_procesamiento/results/<motor>/`. No se modifica la muestra original. Ejecutar **Run All** de arriba abajo.

## Configuración y carga

La carga queda fuera de los tiempos Q1–Q10. Las operaciones perezosas se materializan dentro de cada consulta.

In [1]:
import sys
from pathlib import Path
actual = Path.cwd().resolve()
for candidato in [actual, *actual.parents]:
    fase = candidato / "06_fase6_procesamiento"
    if (fase / "comun.py").is_file():
        sys.path.insert(0, str(fase))
        break
else:
    raise RuntimeError("Ejecutar el notebook dentro del repositorio")
import os
import platform
import numpy as np
import modin.config
modin.config.Engine.put("dask")
import modin.pandas as pd
import pandas as native_pd
from comun import (ENTRADA, ETIQUETAS, PROTOCOLOS, medir, cerrar, filas_tabla)

MOTOR = "modin"
TIEMPOS = []
df = pd.read_csv(ENTRADA)
# Comparación textual exacta para Q1/Q2; los cálculos numéricos usan df.
df_texto = pd.read_csv(ENTRADA, dtype=str, keep_default_na=False)
TOTAL = len(df)
assert TOTAL == 600_000 and len(df.columns) == 40
print("Motor:", MOTOR, "| backend:", modin.config.Engine.get(), "|", platform.platform())
print("Entrada:", ENTRADA, "| registros:", TOTAL)


def familia(datos):
    return datos.assign(Attack_Family=datos["Label"].map(ETIQUETAS))


def tipo(datos):
    return datos.assign(Traffic_Type=datos["Label"].map(
        lambda etiqueta: "Benign" if etiqueta == "Benign" else "Malicious"))


def limpio_rate(datos):
    # +inf -> NaN en Rate solamente: IAT retiene las 600k filas.
    return datos.assign(Rate=datos["Rate"].replace([np.inf, -np.inf], np.nan))




Motor: modin | backend: Dask | Windows-11-10.0.26200-SP0
Entrada: C:\Users\steve\Desktop\Big_Data\ProyectoBigData_CICIoT2023\01_fase1_datos\muestra\CICIoT2023_sample_600k.csv | registros: 600000


## Q1 — Valida las 600.000 filas originales (sin limpiar) y cuenta nulos, infinitos y duplicados exactos.

In [2]:
def q01():
    nulos = int(df.isna().sum().sum())
    nums = df.select_dtypes(include="number")
    infs = int(np.isinf(nums.to_numpy()).sum())
    despues = len(df_texto.drop_duplicates())
    return [{"registros": TOTAL, "columnas": len(df.columns), "nulos": nulos,
             "infinitos": infs, "duplicados": TOTAL - despues}]


r01 = medir("Q1", q01, TIEMPOS, MOTOR)



Q1: 1 filas → q01_validacion.csv


  {'registros': 600000, 'columnas': 40, 'nulos': 18, 'infinitos': 13, 'duplicados': 131030}


Q1: 189.473 s (incluye cómputo/materialización, no exportación)



## Q2 — Deduplicación lógica sobre las 40 columnas; la muestra original permanece intacta.

In [3]:
def q02():
    sin_duplicados = df_texto.drop_duplicates()  # no afecta al original
    despues = len(sin_duplicados)
    return [{"antes": TOTAL, "despues": despues,
             "eliminados": TOTAL - despues,
             "reduccion_pct": (TOTAL - despues) * 100 / TOTAL}]


r02 = medir("Q2", q02, TIEMPOS, MOTOR)



Q2: 1 filas → q02_duplicados.csv


  {'antes': 600000, 'despues': 468970, 'eliminados': 131030, 'reduccion_pct': 21.838333333333335}


Q2: 86.993 s (incluye cómputo/materialización, no exportación)



## Q3 — Transforma Label en una de 8 familias, comprueba la cobertura de las 34 etiquetas.

In [4]:
def q03():
    global clasificado
    clasificado = familia(df)
    t = clasificado.groupby("Attack_Family").agg(
        cantidad=("Label", "size"), clases=("Label", "nunique")).reset_index()
    assert int(t["cantidad"].sum()) == TOTAL and int(t["clases"].sum()) == 34
    assert int(df["Label"].nunique()) == 34
    return filas_tabla(t.sort_values("Attack_Family"))


r03 = medir("Q3", q03, TIEMPOS, MOTOR)



Q3: 8 filas → q03_familias.csv


  {'Attack_Family': 'Benign', 'cantidad': 14087, 'clases': 1}


  {'Attack_Family': 'BruteForce', 'cantidad': 169, 'clases': 1}


  {'Attack_Family': 'DDoS', 'cantidad': 435902, 'clases': 12}


  {'Attack_Family': 'DoS', 'cantidad': 100627, 'clases': 4}


  {'Attack_Family': 'Mirai', 'cantidad': 33788, 'clases': 3}


  {'Attack_Family': 'Recon', 'cantidad': 8862, 'clases': 5}


  {'Attack_Family': 'Spoofing', 'cantidad': 6242, 'clases': 2}


  {'Attack_Family': 'Web-based', 'cantidad': 323, 'clases': 6}


Q3: 32.449 s (incluye cómputo/materialización, no exportación)



## Q4 — Filtra Label != Benign sin reemplazar el dataset original.

In [5]:
def q04():
    n = len(df[df["Label"] != "Benign"])
    return [{"cantidad": n, "porcentaje": n * 100 / TOTAL}]


r04 = medir("Q4", q04, TIEMPOS, MOTOR)



Q4: 1 filas → q04_malicioso.csv


  {'cantidad': 585913, 'porcentaje': 97.65216666666667}


Q4: 13.929 s (incluye cómputo/materialización, no exportación)



## Q5 — Crea Traffic_Type y resume benigno vs malicioso.

In [6]:
def q05():
    global tipado
    tipado = tipo(df)
    t = tipado.groupby("Traffic_Type").size().rename("cantidad").reset_index()
    t["porcentaje"] = t["cantidad"] * 100 / TOTAL
    return filas_tabla(t.sort_values("Traffic_Type"))


r05 = medir("Q5", q05, TIEMPOS, MOTOR)



Q5: 2 filas → q05_tipo_trafico.csv


  {'Traffic_Type': 'Benign', 'cantidad': 14087, 'porcentaje': 2.3478333333333334}


  {'Traffic_Type': 'Malicious', 'cantidad': 585913, 'porcentaje': 97.65216666666667}


Q5: 21.811 s (incluye cómputo/materialización, no exportación)



## Q6 — Agrupa por familia y ordena por cantidad descendente.

In [7]:
def q06():
    t = clasificado.groupby("Attack_Family").size().rename("cantidad").reset_index()
    t["porcentaje"] = t["cantidad"] * 100 / TOTAL
    return filas_tabla(t.sort_values(["cantidad", "Attack_Family"], ascending=[False, True]))


r06 = medir("Q6", q06, TIEMPOS, MOTOR)



Q6: 8 filas → q06_distribucion_familias.csv


  {'Attack_Family': 'DDoS', 'cantidad': 435902, 'porcentaje': 72.65033333333334}


  {'Attack_Family': 'DoS', 'cantidad': 100627, 'porcentaje': 16.771166666666666}


  {'Attack_Family': 'Mirai', 'cantidad': 33788, 'porcentaje': 5.631333333333333}


  {'Attack_Family': 'Benign', 'cantidad': 14087, 'porcentaje': 2.3478333333333334}


  {'Attack_Family': 'Recon', 'cantidad': 8862, 'porcentaje': 1.477}


  {'Attack_Family': 'Spoofing', 'cantidad': 6242, 'porcentaje': 1.0403333333333333}


  {'Attack_Family': 'Web-based', 'cantidad': 323, 'porcentaje': 0.05383333333333333}


  {'Attack_Family': 'BruteForce', 'cantidad': 169, 'porcentaje': 0.028166666666666666}


Q6: 22.496 s (incluye cómputo/materialización, no exportación)



## Q7 — Top 10 de clases maliciosas; porcentaje sobre los 600.000 registros totales.

In [8]:
def q07():
    t = df[df["Label"] != "Benign"].groupby("Label").size().rename("cantidad").reset_index()
    t["porcentaje"] = t["cantidad"] * 100 / TOTAL
    return filas_tabla(t.sort_values(["cantidad", "Label"], ascending=[False, True]).head(10))


r07 = medir("Q7", q07, TIEMPOS, MOTOR)



Q7: 10 filas → q07_top10_ataques.csv


  {'Label': 'DDoS-ICMP_Flood', 'cantidad': 92356, 'porcentaje': 15.392666666666667}


  {'Label': 'DDoS-UDP_Flood', 'cantidad': 69419, 'porcentaje': 11.569833333333333}


  {'Label': 'DDoS-TCP_Flood', 'cantidad': 57689, 'porcentaje': 9.614833333333333}


  {'Label': 'DDoS-PSHACK_FLOOD', 'cantidad': 52521, 'porcentaje': 8.7535}


...

  {'Label': 'DDoS-SynonymousIP_Flood', 'cantidad': 46151, 'porcentaje': 7.691833333333333}


  {'Label': 'DoS-UDP_Flood', 'cantidad': 39416, 'porcentaje': 6.569333333333334}


  {'Label': 'DoS-TCP_Flood', 'cantidad': 34265, 'porcentaje': 5.710833333333333}


  {'Label': 'DoS-SYN_Flood', 'cantidad': 26023, 'porcentaje': 4.337166666666667}


Q7: 36.890 s (incluye cómputo/materialización, no exportación)



## Q8 — Sustituye los 13 Rate infinitos por nulos SOLO en Rate; IAT usa las 600.000 filas.

In [9]:
def q08():
    limpio = limpio_rate(clasificado)
    t = limpio.groupby("Attack_Family").agg(
        registros=("IAT", "size"), validos_rate=("Rate", "count"),
        rate_media=("Rate", "mean"), rate_mediana=("Rate", "median"),
        rate_min=("Rate", "min"), rate_max=("Rate", "max"),
        iat_media=("IAT", "mean"), iat_mediana=("IAT", "median"),
        iat_min=("IAT", "min"), iat_max=("IAT", "max")).reset_index()
    t["rate_inf_excluidos"] = t["registros"] - t["validos_rate"]
    cols=["Attack_Family", "registros", "rate_inf_excluidos", "rate_media", "rate_mediana",
          "rate_min", "rate_max", "iat_media", "iat_mediana", "iat_min", "iat_max"]
    assert int(t["rate_inf_excluidos"].sum()) == 13
    return filas_tabla(t.sort_values("Attack_Family")[cols])


r08 = medir("Q8", q08, TIEMPOS, MOTOR)



Q8: 8 filas → q08_rate_iat_familia.csv


  {'Attack_Family': 'Benign', 'registros': 14087, 'rate_inf_excluidos': 2, 'rate_media': 2667.4811567854676, 'rate_mediana': 171.85122057148476, 'rate_min': 11.18366844132986, 'rate_max': 723155.8620689656, 'iat_media': 0.009185431232298744, 'iat_mediana': 0.0066066026687622, 'iat_min': 1.5020370483398435e-06, 'iat_max': 0.0896720886230468}


  {'Attack_Family': 'BruteForce', 'registros': 169, 'rate_inf_excluidos': 0, 'rate_media': 6106.94353291149, 'rate_mediana': 118.88785271900112, 'rate_min': 9.313311830335037, 'rate_max': 582542.2222222222, 'iat_media': 0.01518694296391048, 'iat_mediana': 0.0094049930572509, 'iat_min': 1.811981201171875e-06, 'iat_max': 0.1103171110153198}


  {'Attack_Family': 'DDoS', 'registros': 435902, 'rate_inf_excluidos': 3, 'rate_media': 32856.65569552071, 'rate_mediana': 29155.456694008062, 'rate_min': 0.0616814565794651, 'rate_max': 7340032.0, 'iat_media': 0.00015632125833893305, 'iat_mediana': 3.471851348876953e-05, 'iat_min': 0.0, 'iat_max': 16.21232791900635}


  {'Attack_Family': 'DoS', 'registros': 100627, 'rate_inf_excluidos': 4, 'rate_media': 22403.689201332756, 'rate_mediana': 21829.41605079629, 'rate_min': 0.000428338067839, 'rate_max': 729444.1739130435, 'iat_media': 0.023857303023602387, 'iat_mediana': 4.632949829101562e-05, 'iat_min': 0.0, 'iat_max': 2334.6046034812925}


  {'Attack_Family': 'Mirai', 'registros': 33788, 'rate_inf_excluidos': 4, 'rate_media': 5615.774688897283, 'rate_mediana': 4628.018781232153, 'rate_min': 0.5485152615709177, 'rate_max': 1677721.6, 'iat_media': 0.00046951247128418214, 'iat_mediana': 0.00021839976310725, 'iat_min': 1.0728836059570312e-06, 'iat_max': 1.8231033301353448}


  {'Attack_Family': 'Recon', 'registros': 8862, 'rate_inf_excluidos': 0, 'rate_media': 11578.112570365392, 'rate_mediana': 107.69648444296658, 'rate_min': 0.0833796569774162, 'rate_max': 2621440.0, 'iat_media': 0.015865826020254728, 'iat_mediana': 0.01047899723052975, 'iat_min': 5.006790161132813e-07, 'iat_max': 11.998069310188294}


  {'Attack_Family': 'Spoofing', 'registros': 6242, 'rate_inf_excluidos': 0, 'rate_media': 31845.4150744199, 'rate_mediana': 639.8389185036615, 'rate_min': 0.0333738298136531, 'rate_max': 1997287.619047619, 'iat_media': 0.011380708923152969, 'iat_mediana': 0.00181715488433835, 'iat_min': 5.006790161132813e-07, 'iat_max': 29.990078592300414}


  {'Attack_Family': 'Web-based', 'registros': 323, 'rate_inf_excluidos': 0, 'rate_media': 6495.370269983226, 'rate_mediana': 79.68629358300149, 'rate_min': 10.03336077914858, 'rate_max': 371177.34513274336, 'iat_media': 0.019921347452760065, 'iat_mediana': 0.014647102355957, 'iat_min': 2.884864807128906e-06, 'iat_max': 0.1025820970535278}


Q8: 26.362 s (incluye cómputo/materialización, no exportación)



## Q9 — Mapea el número IP Protocol Type; no usa las columnas agregadas TCP/UDP/ICMP.

In [10]:
def q09():
    t = clasificado.assign(protocolo=clasificado["Protocol Type"].map(PROTOCOLOS))
    t = t.groupby(["Attack_Family", "Protocol Type", "protocolo"]).size().rename("cantidad").reset_index()
    # 8 filas en el driver para los denominadores, no para los 600k registros.
    total = t.groupby("Attack_Family")["cantidad"].sum().to_dict()
    t["total_familia"] = t["Attack_Family"].map(total)
    t["porcentaje_familia"] = 100 * t["cantidad"] / t["total_familia"]
    return filas_tabla(t.sort_values(["Attack_Family", "Protocol Type"]))


r09 = medir("Q9", q09, TIEMPOS, MOTOR)



Please refer to https://modin.readthedocs.io/en/stable/supported_apis/defaulting_to_pandas.html for explanation.


Q9: 29 filas → q09_protocolos_familia.csv


  {'Attack_Family': 'Benign', 'Protocol Type': 0, 'protocolo': 'HOPOPT', 'cantidad': 31, 'total_familia': 14087, 'porcentaje_familia': 0.22006104919429262}


  {'Attack_Family': 'Benign', 'Protocol Type': 1, 'protocolo': 'ICMP', 'cantidad': 3, 'total_familia': 14087, 'porcentaje_familia': 0.021296230567189607}


  {'Attack_Family': 'Benign', 'Protocol Type': 6, 'protocolo': 'TCP', 'cantidad': 12887, 'total_familia': 14087, 'porcentaje_familia': 91.48150777312415}


...

  {'Attack_Family': 'DDoS', 'Protocol Type': 17, 'protocolo': 'UDP', 'cantidad': 73144, 'total_familia': 435902, 'porcentaje_familia': 16.77991842203064}


  {'Attack_Family': 'DoS', 'Protocol Type': 1, 'protocolo': 'ICMP', 'cantidad': 116, 'total_familia': 100627, 'porcentaje_familia': 0.11527721188150297}


  {'Attack_Family': 'DoS', 'Protocol Type': 6, 'protocolo': 'TCP', 'cantidad': 61228, 'total_familia': 100627, 'porcentaje_familia': 60.846492492074695}


  ... 17 filas adicionales en CSV


Q9: 46.510 s (incluye cómputo/materialización, no exportación)



## Q10 — Compara los dos tipos de tráfico. Rate excluye 13 infinitos; las demás métricas no.

In [11]:
def q10():
    limpio = limpio_rate(tipado)
    t = limpio.groupby("Traffic_Type").agg(
        registros=("IAT", "size"), validos_rate=("Rate", "count"),
        rate_media=("Rate", "mean"), rate_mediana=("Rate", "median"),
        iat_media=("IAT", "mean"), iat_mediana=("IAT", "median"),
        ack_flag_media=("ack_flag_number", "mean"),
        syn_flag_media=("syn_flag_number", "mean"),
        tot_sum_media=("Tot sum", "mean"), avg_media=("AVG", "mean")).reset_index()
    t["rate_inf_excluidos"] = t["registros"] - t["validos_rate"]
    cols=["Traffic_Type", "registros", "rate_inf_excluidos", "rate_media",
          "rate_mediana", "iat_media", "iat_mediana", "ack_flag_media",
          "syn_flag_media", "tot_sum_media", "avg_media"]
    assert int(t["rate_inf_excluidos"].sum()) == 13
    return filas_tabla(t.sort_values("Traffic_Type")[cols])


r10 = medir("Q10", q10, TIEMPOS, MOTOR)



Q10: 2 filas → q10_perfil_trafico.csv


  {'Traffic_Type': 'Benign', 'registros': 14087, 'rate_inf_excluidos': 2, 'rate_media': 2667.4811567854676, 'rate_mediana': 171.85122057148476, 'iat_media': 0.009185431232298744, 'iat_mediana': 0.0066066026687622, 'ack_flag_media': 0.8030792771901596, 'syn_flag_media': 0.013679278767658126, 'tot_sum_media': 6088.058777596366, 'avg_media': 609.1035888092252}


  {'Traffic_Type': 'Malicious', 'registros': 585913, 'rate_inf_excluidos': 11, 'rate_media': 29135.840983926613, 'rate_mediana': 25247.11972551616, 'iat_media': 0.004617299823977632, 'iat_mediana': 4.008054733276368e-05, 'ack_flag_media': 0.11434509125214834, 'syn_flag_media': 0.2121034588419937, 'tot_sum_media': 11095.438104291934, 'avg_media': 120.07703692400342}


Q10: 26.547 s (incluye cómputo/materialización, no exportación)



## Tiempos por consulta

Tiempos de pared (segundos). Comparar solo con atención a carga, caché, hilos y materialización.

In [12]:
cerrar(MOTOR, TIEMPOS)


Tiempos medidos: [{'Consulta': 'Q1', 'Tiempo': 189.473}, {'Consulta': 'Q2', 'Tiempo': 86.993}, {'Consulta': 'Q3', 'Tiempo': 32.449}, {'Consulta': 'Q4', 'Tiempo': 13.929}, {'Consulta': 'Q5', 'Tiempo': 21.811}, {'Consulta': 'Q6', 'Tiempo': 22.496}, {'Consulta': 'Q7', 'Tiempo': 36.89}, {'Consulta': 'Q8', 'Tiempo': 26.362}, {'Consulta': 'Q9', 'Tiempo': 46.51}, {'Consulta': 'Q10', 'Tiempo': 26.547}]
